# Reproducible ASEAN same-period analysis

This notebook answers a concrete TradeGravity task using only published schema 2.0 JSON. Values are nominal current US dollars. Always retain period and provider fields when reusing the output.

In [ ]:
import csv, json
from pathlib import Path
from urllib.request import urlopen

BASE = "https://elecpapaya.github.io/TradeGravity/data"
def load_json(path):
    with urlopen(f"{BASE}/{path}", timeout=30) as response:
        return json.load(response)

meta = load_json("meta.json")
latest = load_json("latest.json")
series = load_json("series.json")
assert meta["schema_version"].startswith("2.")
print({k: meta[k] for k in ["schema_version", "generated_at", "provider", "dominant_period"]})

## Rank ASEAN reporters on an exact, comparable 2023 point

The context group comes from `latest.json`; the requested historical point comes from `series.json`. A reporter is included only when both partner blocks are available for that point.

In [ ]:
context = {row["iso3"]: row for row in latest["rows"]}
ranking = []
for reporter in series["rows"]:
    country = context.get(reporter["iso3"], {})
    if "ASEAN" not in country.get("groups", []):
        continue
    point = next((p for p in reporter["points"] if p["period_type"] == "Y" and p["period"] == "2023" and p["comparable"]), None)
    if point is None:
        continue
    ranking.append({
        "iso3": reporter["iso3"], "name": country.get("name", reporter["iso3"]),
        "period": point["period"], "usa_trade_usd": point["usa"]["trade"],
        "chn_trade_usd": point["chn"]["trade"], "combined_trade_usd": point["total"],
        "china_share": point["share_cn"], "provider": meta["provider"],
    })
ranking.sort(key=lambda row: row["combined_trade_usd"], reverse=True)
ranking

In [ ]:
output = Path("asean-2023-tradegravity.csv")
if ranking:
    with output.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=ranking[0].keys())
        writer.writeheader(); writer.writerows(ranking)
print(output.resolve(), len(ranking), "comparable ASEAN reporters")

## Inspect Viet Nam's trend and HS2 product mix

Product provenance is printed separately because it may differ from the headline provider.

In [ ]:
selected = "VNM"
selected_series = next(row for row in series["rows"] if row["iso3"] == selected)
annual = [p for p in selected_series["points"] if p["period_type"] == "Y" and p["comparable"]]
trend = [(p["period"], p["usa"]["trade"], p["chn"]["trade"]) for p in annual]
trend

In [ ]:
products = load_json(f"products/{selected}.json")
product_period = products["periods"][0]
top_products = sorted(
    (row for row in products["rows"] if row["period"] == product_period),
    key=lambda row: row["total"], reverse=True
)[:10]
print(products["provider"], products["classification"], products["level"], product_period)
[(row["code"], row["name"], row["total"]) for row in top_products]